<a href="https://colab.research.google.com/github/Monicababu21/Text-Similarity-Analyzer/blob/main/Text_Similarity_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install gradio nltk pandas scikit-learn

In [2]:
import gradio as gr
import nltk
import pandas as pd
import re
import tempfile

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words("english"))

print("Text Similarity Analyzer is ready!")

Text Similarity Analyzer is ready!


In [3]:
def clean_text(text):
    if not text:
        return ""

    text = text.lower()

    # Remove punctuation
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def get_words(text):
    cleaned = clean_text(text)

    words = cleaned.split()

    return [
        word
        for word in words
        if word not in STOP_WORDS and len(word) > 2
    ]


print("Text preprocessing completed.")

Text preprocessing completed.


In [4]:
def jaccard_similarity(text1, text2):

    words1 = set(get_words(text1))
    words2 = set(get_words(text2))

    if not words1 and not words2:
        return 100.0

    if not words1 or not words2:
        return 0.0

    intersection = words1.intersection(words2)
    union = words1.union(words2)

    score = len(intersection) / len(union)

    return round(score * 100, 2)


print("Jaccard similarity ready.")

Jaccard similarity ready.


In [5]:
def cosine_similarity_score(text1, text2):

    if not text1.strip() or not text2.strip():
        return 0.0

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english"
        )

        matrix = vectorizer.fit_transform([
            text1,
            text2
        ])

        score = cosine_similarity(
            matrix[0:1],
            matrix[1:2]
        )[0][0]

        return round(score * 100, 2)

    except Exception:
        return 0.0


print("Cosine similarity ready.")

Cosine similarity ready.


In [6]:
def text_statistics(text):

    if not text or not text.strip():
        return {
            "words": 0,
            "characters": 0,
            "sentences": 0,
            "paragraphs": 0,
            "unique_words": 0
        }

    words = re.findall(
        r"\b[\w']+\b",
        text
    )

    sentences = [
        s.strip()
        for s in re.split(
            r"[.!?]+",
            text
        )
        if s.strip()
    ]

    paragraphs = [
        p.strip()
        for p in re.split(
            r"\n\s*\n",
            text
        )
        if p.strip()
    ]

    unique_words = set(
        word.lower()
        for word in words
    )

    return {
        "words": len(words),
        "characters": len(text),
        "sentences": len(sentences),
        "paragraphs": len(paragraphs),
        "unique_words": len(unique_words)
    }


print("Text statistics ready.")

Text statistics ready.


In [7]:
def compare_words(text1, text2):

    words1 = set(get_words(text1))
    words2 = set(get_words(text2))

    common_words = sorted(
        words1.intersection(words2)
    )

    unique_text1 = sorted(
        words1 - words2
    )

    unique_text2 = sorted(
        words2 - words1
    )

    return (
        common_words,
        unique_text1,
        unique_text2
    )


print("Word comparison ready.")

Word comparison ready.


In [8]:
def sentence_similarity(sentence1, sentence2):

    words1 = set(get_words(sentence1))
    words2 = set(get_words(sentence2))

    if not words1 or not words2:
        return 0

    intersection = words1.intersection(words2)
    union = words1.union(words2)

    return len(intersection) / len(union)


def find_matching_sentences(text1, text2):

    sentences1 = [
        s.strip()
        for s in re.split(
            r"[.!?]+",
            text1
        )
        if s.strip()
    ]

    sentences2 = [
        s.strip()
        for s in re.split(
            r"[.!?]+",
            text2
        )
        if s.strip()
    ]

    matches = []

    for sentence1 in sentences1:

        best_score = 0
        best_sentence = ""

        for sentence2 in sentences2:

            score = sentence_similarity(
                sentence1,
                sentence2
            )

            if score > best_score:
                best_score = score
                best_sentence = sentence2

        if best_score >= 0.60:

            matches.append({
                "Text A": sentence1,
                "Text B": best_sentence,
                "Similarity": round(
                    best_score * 100,
                    2
                )
            })

    return matches


print("Sentence comparison ready.")

Sentence comparison ready.


In [9]:
def classify_similarity(score):

    if score >= 80:
        return "Very High Similarity"

    elif score >= 60:
        return "High Similarity"

    elif score >= 40:
        return "Moderate Similarity"

    elif score >= 20:
        return "Low Similarity"

    else:
        return "Very Low Similarity"


print("Similarity classification ready.")

Similarity classification ready.


In [10]:
def analyze_texts(text1, text2):

    if not text1 or not text1.strip():
        return (
            "Please enter Text A.",
            "",
            None
        )

    if not text2 or not text2.strip():
        return (
            "Please enter Text B.",
            "",
            None
        )

    # Similarity scores
    jaccard = jaccard_similarity(
        text1,
        text2
    )

    cosine = cosine_similarity_score(
        text1,
        text2
    )

    # Combined similarity
    final_score = round(
        (jaccard + cosine) / 2,
        2
    )

    classification = classify_similarity(
        final_score
    )

    # Statistics
    stats1 = text_statistics(text1)
    stats2 = text_statistics(text2)

    # Words
    common_words, unique1, unique2 = compare_words(
        text1,
        text2
    )

    # Sentences
    matching_sentences = find_matching_sentences(
        text1,
        text2
    )

    # --------------------------------------------------
    # Result
    # --------------------------------------------------

    result = f"""
# Text Similarity Result

## Similarity Score: **{final_score}%**

### Classification: **{classification}**

| Method | Score |
|---|---:|
| Jaccard Similarity | {jaccard}% |
| Cosine Similarity | {cosine}% |
| Combined Score | {final_score}% |

---

## Text Statistics

| Metric | Text A | Text B |
|---|---:|---:|
| Words | {stats1['words']} | {stats2['words']} |
| Characters | {stats1['characters']} | {stats2['characters']} |
| Sentences | {stats1['sentences']} | {stats2['sentences']} |
| Paragraphs | {stats1['paragraphs']} | {stats2['paragraphs']} |
| Unique Words | {stats1['unique_words']} | {stats2['unique_words']} |

---

## Common Words

**{len(common_words)} common words**

{", ".join(common_words[:50]) if common_words else "No common words found."}

---

## Words Only in Text A

{", ".join(unique1[:50]) if unique1 else "None"}

---

## Words Only in Text B

{", ".join(unique2[:50]) if unique2 else "None"}

---

## Matching Sentences

**{len(matching_sentences)} similar sentence(s) detected.**
"""

    for i, match in enumerate(
        matching_sentences[:10],
        start=1
    ):

        result += f"""

### Match {i}

**Text A:**
{match['Text A']}

**Text B:**
{match['Text B']}

**Similarity:** {match['Similarity']}%
"""

    # --------------------------------------------------
    # Report
    # --------------------------------------------------

    report = f"""
TEXT SIMILARITY ANALYZER
========================

Overall Similarity: {final_score}%
Classification: {classification}

Jaccard Similarity: {jaccard}%
Cosine Similarity: {cosine}%

TEXT A
------
Words: {stats1['words']}
Characters: {stats1['characters']}
Sentences: {stats1['sentences']}
Paragraphs: {stats1['paragraphs']}
Unique Words: {stats1['unique_words']}

TEXT B
------
Words: {stats2['words']}
Characters: {stats2['characters']}
Sentences: {stats2['sentences']}
Paragraphs: {stats2['paragraphs']}
Unique Words: {stats2['unique_words']}

COMMON WORDS
------------
{", ".join(common_words)}

MATCHING SENTENCES
------------------
"""

    for i, match in enumerate(
        matching_sentences,
        start=1
    ):

        report += f"""

Match {i}

Text A:
{match['Text A']}

Text B:
{match['Text B']}

Similarity:
{match['Similarity']}%
"""

    report_file = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".txt",
        mode="w",
        encoding="utf-8"
    )

    report_file.write(report)
    report_file.close()

    return (
        result,
        classification,
        report_file.name
    )


print("Main analysis function ready.")

Main analysis function ready.


In [12]:
# ============================================================
# CELL 12 — FINAL GRADIO INTERFACE
# ============================================================

import gradio as gr


# ------------------------------------------------------------
# SAMPLE TEXTS
# ------------------------------------------------------------

SAMPLES = {
    "Technology": (
        """
        Artificial intelligence is transforming modern technology.
        Machine learning systems can analyze large amounts of data
        and identify useful patterns.
        """,

        """
        Artificial intelligence is changing modern technology.
        Machine learning systems can process large amounts of data
        and identify important patterns.
        """
    ),

    "Education": (
        """
        Online education allows students to access learning materials
        from anywhere. Digital platforms provide flexibility and
        interactive learning experiences.
        """,

        """
        Digital learning platforms allow students to study
        from different locations. Online education provides
        flexibility and interactive educational experiences.
        """
    ),

    "Different": (
        """
        Renewable energy sources such as solar and wind power
        can reduce dependence on fossil fuels.
        """,

        """
        Computer networks allow devices to communicate
        and share information across different locations.
        """
    )
}


# ------------------------------------------------------------
# LOAD SAMPLE FUNCTION
# ------------------------------------------------------------

def load_sample(name):

    if name not in SAMPLES:
        return "", ""

    text1, text2 = SAMPLES[name]

    return text1.strip(), text2.strip()


# ------------------------------------------------------------
# CUSTOM CSS
# ------------------------------------------------------------

custom_css = """

body {
    background: #ffffff;
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto;
}

h1, h2, h3 {
    color: #6d28d9;
}

button.primary {
    background: #7c3aed !important;
    border: none !important;
}

button.primary:hover {
    background: #6d28d9 !important;
}

textarea {
    border-radius: 12px !important;
}

"""


# ------------------------------------------------------------
# GRADIO APPLICATION
# ------------------------------------------------------------

with gr.Blocks(
    title="Text Similarity Analyzer"
) as app:

    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        """
        # Text Similarity Analyzer

        ### Compare two texts and analyze their similarity using NLP techniques.

        **Jaccard Similarity • Cosine Similarity • Word Comparison • Sentence Matching**
        """
    )


    # ========================================================
    # TAB 1 — COMPARE TEXTS
    # ========================================================

    with gr.Tab("Compare Texts"):

        with gr.Row():

            with gr.Column():

                text_a = gr.Textbox(
                    label="Text A",
                    placeholder="Enter or paste the first text...",
                    lines=14
                )

            with gr.Column():

                text_b = gr.Textbox(
                    label="Text B",
                    placeholder="Enter or paste the second text...",
                    lines=14
                )


        # ----------------------------------------------------
        # SAMPLE TEXTS
        # ----------------------------------------------------

        sample_selector = gr.Dropdown(
            choices=[
                "Technology",
                "Education",
                "Different"
            ],
            label="Load Sample"
        )

        load_button = gr.Button(
            "Load Sample"
        )

        load_button.click(
            fn=load_sample,
            inputs=sample_selector,
            outputs=[
                text_a,
                text_b
            ]
        )


        # ----------------------------------------------------
        # COMPARE BUTTON
        # ----------------------------------------------------

        compare_button = gr.Button(
            "Compare Texts",
            variant="primary"
        )


        # ----------------------------------------------------
        # OUTPUTS
        # ----------------------------------------------------

        result_output = gr.Markdown()

        classification_output = gr.Markdown()

        report_file = gr.File(
            label="Download Analysis Report"
        )


        # ----------------------------------------------------
        # ANALYSIS EVENT
        # ----------------------------------------------------

        compare_button.click(
            fn=analyze_texts,
            inputs=[
                text_a,
                text_b
            ],
            outputs=[
                result_output,
                classification_output,
                report_file
            ]
        )


    # ========================================================
    # TAB 2 — HOW IT WORKS
    # ========================================================

    with gr.Tab("How It Works"):

        gr.Markdown(
            """
            # How Text Similarity Analysis Works

            ### 1. Text Preprocessing

            The system converts text to lowercase, removes
            punctuation and removes common stopwords.

            ### 2. Jaccard Similarity

            Measures the overlap between the unique words
            in Text A and Text B.

            ### 3. TF-IDF Cosine Similarity

            Converts text into numerical vectors and
            calculates the similarity between them.

            ### 4. Sentence Matching

            Individual sentences are compared to identify
            potentially similar sentences.

            ### 5. Similarity Classification

            | Score | Classification |
            |---|---|
            | 80–100% | Very High Similarity |
            | 60–79% | High Similarity |
            | 40–59% | Moderate Similarity |
            | 20–39% | Low Similarity |
            | 0–19% | Very Low Similarity |

            **Note:** A similarity score indicates textual
            similarity. It does not automatically prove plagiarism.
            """
        )


    # ========================================================
    # TAB 3 — ABOUT
    # ========================================================

    with gr.Tab("About"):

        gr.Markdown(
            """
            # About the Project

            **Text Similarity Analyzer** is a Text Analysis
            and Processing project that compares two pieces
            of text using Natural Language Processing techniques.

            ## Features

            - Text comparison
            - Similarity percentage
            - Jaccard similarity
            - TF-IDF cosine similarity
            - Common word detection
            - Unique word detection
            - Matching sentence detection
            - Text statistics
            - Similarity classification
            - Downloadable analysis report

            ## Technologies

            - Python
            - Gradio
            - NLTK
            - Pandas
            - Scikit-learn
            - Google Colab
            """
        )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="purple"
    ),
    css=custom_css
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9ee4db6d774ee9106c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
